In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mc
from scipy import stats
import statsmodels.stats.multitest as multitest
import warnings

warnings.filterwarnings('ignore')

# =============================================================================
# 1. Settings & Color Blending
# =============================================================================
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
DPI_SETTING = 600

def clean_and_convert(vals):
    s_vals = pd.Series(vals).astype(str).str.strip()
    s_vals = s_vals.replace(['Undetermined', '-', 'nan', 'NaN', '#VALUE!', ''], '0')
    return pd.to_numeric(s_vals, errors='coerce').fillna(0)

def lighten_hex(hex_color, factor=0.3):
    """指定された16進数の色を、係数(0.0〜1.0)の割合だけ白に近づける（明度を上げる）"""
    r, g, b = mc.to_rgb(hex_color)
    r = r + (1 - r) * factor
    g = g + (1 - g) * factor
    b = b + (1 - b) * factor
    return mc.to_hex((r, g, b))

# Original base colors extracted strictly from @4cd.ipynb HIGHLIGHTS
base_colors = {
    'RS': '#00CED1',           # Turquoise
    'Xylitol': '#008000',      # Green
    'Erythritol': '#FF0000'    # Red
}

# Apply 30% lightening to make them slightly softer
palette_dict = {k: lighten_hex(v, 0.3) for k, v in base_colors.items()}

name_map = {
    'Resistant starch': 'RS',
    'Xylitol': 'Xylitol',
    'Erythritol': 'Erythritol'
}

# =============================================================================
# 2. Data Loading & Correlation
# =============================================================================
df_ace = pd.read_csv('Acetate(mM).csv')
df_but = pd.read_csv('Butyrate(mM).csv')
df_pro = pd.read_csv('Propionate(mM).csv')

donor_cols = [c for c in df_but.columns if c.startswith('HS-')]

# Get Control baselines
ctrl_ace = clean_and_convert(df_ace[df_ace['KULFFI'].str.strip() == 'Control'][donor_cols].iloc[0])
ctrl_but = clean_and_convert(df_but[df_but['KULFFI'].str.strip() == 'Control'][donor_cols].iloc[0])
ctrl_pro = clean_and_convert(df_pro[df_pro['KULFFI'].str.strip() == 'Control'][donor_cols].iloc[0])

def plot_5bcd(substrate_raw, output_name):
    short_name = name_map[substrate_raw]
    color = palette_dict[short_name]

    sub_ace = clean_and_convert(df_ace[df_ace['KULFFI'].str.strip() == substrate_raw][donor_cols].iloc[0])
    sub_but = clean_and_convert(df_but[df_but['KULFFI'].str.strip() == substrate_raw][donor_cols].iloc[0])
    sub_pro = clean_and_convert(df_pro[df_pro['KULFFI'].str.strip() == substrate_raw][donor_cols].iloc[0])

    delta_ace = sub_ace - ctrl_ace
    delta_but = sub_but - ctrl_but
    delta_pro = sub_pro - ctrl_pro

    # Calculate P-values for the 3 SCFA combinations
    r_ab, p_ab = stats.pearsonr(delta_ace, delta_but)
    r_ap, p_ap = stats.pearsonr(delta_ace, delta_pro)
    r_pb, p_pb = stats.pearsonr(delta_pro, delta_but)

    # FDR Correction across the 3 SCFA correlation pairs
    p_vals = [p_ab, p_ap, p_pb]
    _, q_vals, _, _ = multitest.multipletests(p_vals, alpha=0.05, method='fdr_bh')
    q_ab = q_vals[0] # Extract the q-value for Acetate vs Butyrate

    df_plot = pd.DataFrame({'Delta_Acetate': delta_ace.values, 'Delta_Butyrate': delta_but.values})

    # =============================================================================
    # 3. Figure Generation
    # =============================================================================
    fig, ax = plt.subplots(figsize=(5, 5))

    # Condition: Solid black line if absolute correlation is >= 0.6
    if abs(r_ab) >= 0.6:
        line_kws = {'linewidth': 2, 'color': 'black', 'alpha': 1.0, 'linestyle': '-'}
    else:
        line_kws = {'linewidth': 2, 'color': 'gray', 'alpha': 0.5, 'linestyle': '-'}

    sns.regplot(x='Delta_Acetate', y='Delta_Butyrate', data=df_plot, ax=ax, color=color,
                scatter_kws={'s': 60, 'alpha': 0.7, 'edgecolors': 'white', 'linewidths': 0.5},
                line_kws=line_kws)

    ax.axhline(0, color='black', linestyle=':', linewidth=1.5)
    ax.axvline(0, color='black', linestyle=':', linewidth=1.5)

    ax.set_xlabel(r'$\Delta$ Acetate (mM)', fontsize=14, fontweight='bold', labelpad=10)
    ax.set_ylabel(r'$\Delta$ Butyrate (mM)', fontsize=14, fontweight='bold', labelpad=10)

    ax.tick_params(axis='both', labelsize=12, width=1.5, length=5)
    for spine in ['left', 'bottom']: ax.spines[spine].set_linewidth(1.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_box_aspect(1)

    # Stats label
    q_text = "q < 0.001" if q_ab < 0.001 else f"q = {q_ab:.3f}"
    stats_text = f"$r = {r_ab:.2f}$\n{q_text}"

    ax.text(0.05, 0.95, stats_text, transform=ax.transAxes,
            fontsize=14, fontweight='normal', va='top', ha='left', color='black',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=1.0, edgecolor='none'))

    plt.tight_layout()
    plt.savefig(f'{output_name}.pdf', dpi=DPI_SETTING, bbox_inches='tight', transparent=True)
    plt.close()

# Execute plot generation
plot_5bcd('Resistant starch', 'Figure_5b')
plot_5bcd('Xylitol', 'Figure_5c')
plot_5bcd('Erythritol', 'Figure_5d')